In [25]:
import numpy as np
import pandas as pd
import sns
from pandas.core.common import random_state
from scipy.stats import alpha
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

import training
import importlib
import mlflow
from sklearn.linear_model import LogisticRegression, LinearRegression
import dagshub
import ML_Assignment_1.preprocessor as pre


from ML_Assignment_1.mapping import ORDINAL_COLUMNS
from ML_Assignment_1.preprocessor import WOEEncoder

importlib.reload(training)
importlib.reload(pre)
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

In [26]:
dagshub.init(repo_owner='lukaLomadze', repo_name='ML_Assignment_1', mlflow=True)

Initialized MLflow to track repo "lukaLomadze/ML_Assignment_1"

Repository lukaLomadze/ML_Assignment_1 initialized!

In [27]:
train= pd.read_csv("../houses/train.csv")
test= pd.read_csv("../houses/test.csv")
test_ids=test["Id"]

In [28]:
missing = train.isna().mean()
high_missing = missing[missing>0.70].index.tolist()
is_outlier= (train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)
clean_train = train.loc[~is_outlier].reset_index(drop=True)
train = clean_train.drop(columns=high_missing)
test = test.drop(columns=[c for c in high_missing if c in test.columns])


In [29]:
y = np.log1p(train['SalePrice'])
x=train.drop(columns=['SalePrice',"Id"])
xTest= test.drop(columns=['Id'])

In [30]:
print(y.shape, x.shape, xTest.shape)

(1458,) (1458, 75) (1459, 75)


In [31]:
from mapping import ORDINAL_COLUMNS
filler = pre.NAFiller()
qual = pre.QualityEncoder(ORDINAL_COLUMNS=ORDINAL_COLUMNS)
feat = pre.FeatureAdder()

x= feat.fit_transform(qual.fit_transform(filler.fit_transform(x,y)))
xTest = feat.transform(qual.transform(filler.transform(xTest)))

In [32]:
cat_cols= x.select_dtypes(include="object").columns
cardinal= x[cat_cols].nunique()
woe_cols = cardinal[cardinal>5].index.tolist()

C:\Users\lukas\AppData\Local\Temp\ipykernel_8528\1361995879.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols= x.select_dtypes(include="object").columns


In [33]:
woe_enc = pre.WOEEncoder(columns=woe_cols)
x=woe_enc.fit_transform(x,y)
xTest=woe_enc.transform(xTest)

In [34]:
ohe = pre.OneHotEncoderSafe()
x=ohe.fit_transform(x,y)
xTest=ohe.transform(xTest)


In [35]:
scaler = StandardScaler()
x = pd.DataFrame(scaler.fit_transform(x), columns=x.columns)
xTest = pd.DataFrame(scaler.transform(xTest), columns=x.columns)

In [36]:
from sklearn.feature_selection import RFE

rfe = RFE(estimator=Ridge(), n_features_to_select=30)
rfe = rfe.fit(x, y)
rfe_cols= x.columns[rfe.support_].tolist()
x= x[rfe_cols]
xTest = xTest[rfe_cols]


In [37]:
corr = x.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop= set()
for col in upper.columns:
    for p in upper.index[upper[col]>0.85].tolist():
        dcol = p if abs(x[col].corr(y))>= abs(x[p].corr(y)) else col
        drop.add(dcol)
corr_cols= [c for c in rfe_cols if c not in drop]
x = x[corr_cols]
xTest = xTest[corr_cols]


survuved features: 


In [38]:
print("survuved features: ", x.columns)

survuved features:  Index(['LotArea', 'OverallQual', 'OverallCond', 'BsmtQual', 'BsmtExposure',
       'BsmtFinSF1', 'TotalBsmtSF', 'HeatingQC', '1stFlrSF', '2ndFlrSF',
       'KitchenAbvGr', 'KitchenQual', 'Functional', 'Fireplaces', 'GarageCars',
       'ScreenPorch', 'TotalSF', 'TotalBathrooms', 'HouseAge', 'HasPool',
       'Neighborhood_woe', 'Foundation_woe', 'SaleCondition_woe',
       'MSZoning_FV', 'MSZoning_RH', 'MSZoning_RL', 'MSZoning_RM'],
      dtype='str')


In [41]:
model = mlflow.sklearn.load_model('models:/house-prices-best/Production')
print('Loaded:', type(model).__name__)

MlflowException: No versions of model with name 'house-prices-best' and stage 'Production' found

In [ ]:
model.fit(x,y)
pred= np.expm1(model.predict(xTest))
sub= pd.DataFrame({'Id': test_ids, 'SalePrice': pred})
sub.to_csv('submission.csv', index=False)
print("file saved")
print(sub)